# Unsupervised Candlestick Pattern Discovery with K-Means

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/unsupervised/candlestick_clustering.ipynb)

Discover candlestick patterns objectively using K-Means clustering on OHLC data.

**Blog post:** [Unsupervised Candlestick Pattern Discovery with K-Means](https://sesen.ai/blog/candlestick-pattern-discovery-kmeans-clustering)

**Key references:**
- Lloyd, S. (1982). Least squares quantization in PCM. *IEEE Trans. Information Theory*, 28(2), 129-137.
- Nison, S. (1991). *Japanese Candlestick Charting Techniques*. New York Institute of Finance.

In [ ]:
!pip install -q yfinance mplfinance

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

np.random.seed(42)

## 1. Fetch Data and Engineer Features

In [ ]:
df = yf.Ticker("SPY").history(period="10y", interval="1d")
df = df[['Open', 'High', 'Low', 'Close']].dropna()

# Feature engineering (faithful to original R code)
df['body']  = df['Close'] - df['Open']
df['upper'] = df['High']  - df['Open']
df['lower'] = df['Low']   - df['Open']
df['range'] = df['High']  - df['Low']
df['ma_10'] = df['Close'].rolling(10).mean()
df['ma_5']  = df['Close'].rolling(5).mean()
df['trend'] = df['ma_10'] - df['ma_5']
df['vol']   = df['Close'].rolling(10).std()
df = df.dropna()
print(f"{len(df)} daily bars: {df.index[0].date()} to {df.index[-1].date()}")

## 2. Single-Candle Clustering

In [ ]:
features = ['body', 'upper', 'lower', 'trend', 'vol']
scaler = StandardScaler()
X = scaler.fit_transform(df[features])

# Elbow method
inertias = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(X).inertia_
            for k in range(1, 13)]

plt.figure(figsize=(8, 4))
plt.plot(range(1, 13), inertias, 'o-', color='#2196F3', linewidth=2, markersize=8)
plt.axvline(x=5, color='#FF9800', linestyle='--', alpha=0.7, label='k = 5')
plt.xlabel('k'); plt.ylabel('WCSS')
plt.title('Elbow Method: Single-Candle Features')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
km = KMeans(n_clusters=5, n_init=10, random_state=42)
df['cluster'] = km.fit_predict(X)

labels = {0: 'Quiet Day', 1: 'Volatile Drift', 2: 'Surge Rally',
          3: 'Sharp Selloff', 4: 'Steady Bull'}
for c in range(5):
    mask = df['cluster'] == c
    n = mask.sum()
    print(f"Cluster {c} ({labels[c]:17s}): n={n:4d} ({n/len(df)*100:.0f}%), "
          f"body=${df.loc[mask,'body'].mean():+.2f}, vol=${df.loc[mask,'vol'].mean():.2f}")

## 3. Colour-Coded Candlestick Chart

In [ ]:
palette = ['#9E9E9E', '#FF9800', '#4CAF50', '#E53935', '#2196F3']
recent = df.tail(100).copy()
recent['idx'] = range(len(recent))

fig, ax = plt.subplots(figsize=(14, 5))
for _, row in recent.iterrows():
    i, o, h, l, c, cl = row['idx'], row['Open'], row['High'], row['Low'], row['Close'], int(row['cluster'])
    body_lo, body_hi = min(o, c), max(o, c)
    ax.bar(i, body_hi - body_lo, bottom=body_lo, width=0.6, color=palette[cl], edgecolor=palette[cl])
    ax.plot([i, i], [l, body_lo], color='black', linewidth=0.8)
    ax.plot([i, i], [body_hi, h], color='black', linewidth=0.8)

from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=palette[k], label=labels[k]) for k in range(5)], loc='upper left')
tick_pos = list(range(0, len(recent), 20))
ax.set_xticks(tick_pos)
ax.set_xticklabels([recent.index[i].strftime('%b %Y') for i in tick_pos])
ax.set_ylabel('SPY Price ($)'); ax.set_title('SPY Candlesticks by K-Means Cluster')
ax.grid(True, alpha=0.2, axis='y'); fig.tight_layout(); plt.show()

## 4. Transition Matrix

In [ ]:
seq = df['cluster'].values
trans = np.zeros((5, 5))
for i in range(len(seq) - 1):
    trans[seq[i], seq[i+1]] += 1
trans_prob = trans / trans.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(trans_prob, cmap='YlOrRd', vmin=0, vmax=0.7)
short = ['Quiet', 'Vol Drift', 'Surge', 'Selloff', 'Bull']
for i in range(5):
    for j in range(5):
        ax.text(j, i, f'{trans_prob[i,j]:.2f}', ha='center', va='center',
                fontsize=11, fontweight='bold', color='white' if trans_prob[i,j] > 0.4 else 'black')
ax.set_xticks(range(5)); ax.set_yticks(range(5))
ax.set_xticklabels(short, rotation=30, ha='right'); ax.set_yticklabels(short)
ax.set_xlabel('Next State'); ax.set_ylabel('Current State')
ax.set_title('Cluster Transition Probabilities')
plt.colorbar(im, ax=ax, shrink=0.8); fig.tight_layout(); plt.show()

## 5. Multi-Candle Patterns: Triplets

In [ ]:
# Proportional features for multi-candle windows
df['body_pct']  = (df['Close'] - df['Open']) / df['range']
df['upper_pct'] = (df['High'] - df[['Open','Close']].max(axis=1)) / df['range']
df['lower_pct'] = (df[['Open','Close']].min(axis=1) - df['Low']) / df['range']

def build_window_features(df, window_size):
    feats, indices = [], []
    for i in range(window_size - 1, len(df)):
        window = df.iloc[i - window_size + 1 : i + 1]
        row = []
        for j, (_, candle) in enumerate(window.iterrows()):
            row.extend([candle['body_pct'], candle['upper_pct'], candle['lower_pct']])
            if j > 0:
                prev = window.iloc[j - 1]
                avg_range = (candle['range'] + prev['range']) / 2
                gap = (candle['Open'] - prev['Close']) / avg_range if avg_range > 0 else 0
                rr = candle['range'] / prev['range'] if prev['range'] > 0 else 1
                row.extend([gap, rr])
        feats.append(row)
        indices.append(df.index[i])
    return np.array(feats), indices

X_trip, trip_idx = build_window_features(df, 3)
X_trip_scaled = StandardScaler().fit_transform(X_trip)
trip_labels = KMeans(n_clusters=6, n_init=10, random_state=42).fit_predict(X_trip_scaled)
print(f"Triplet windows: {len(X_trip)}, features: {X_trip.shape[1]}")
for c in range(6):
    mask = trip_labels == c
    bodies = [X_trip[mask, j*3].mean() for j in range(3)]
    print(f"  Cluster {c}: n={mask.sum():4d}, bodies: {' -> '.join(f'{b:+.2f}' for b in bodies)}")

## 6. Multi-Candle Patterns: Quadruplets

In [ ]:
X_quad, quad_idx = build_window_features(df, 4)
X_quad_scaled = StandardScaler().fit_transform(X_quad)
quad_labels = KMeans(n_clusters=8, n_init=10, random_state=42).fit_predict(X_quad_scaled)
print(f"Quadruplet windows: {len(X_quad)}, features: {X_quad.shape[1]}")
for c in range(8):
    mask = quad_labels == c
    bodies = [X_quad[mask, j*3].mean() for j in range(4)]
    print(f"  Cluster {c}: n={mask.sum():4d}, bodies: {' -> '.join(f'{b:+.2f}' for b in bodies)}")

## 7. Predictive Analysis: Do Patterns Forecast Returns?

In [ ]:
df['next_ret'] = df['Close'].pct_change().shift(-1) * 100

print("Next-day return after each triplet pattern:")
for c in range(6):
    mask = trip_labels == c
    idx = [trip_idx[i] for i in range(len(trip_labels)) if mask[i]]
    rets = df.loc[idx, 'next_ret'].dropna()
    mean_r = rets.mean()
    t = mean_r / (rets.std() / np.sqrt(len(rets)))
    print(f"  Cluster {c}: mean={mean_r:+.4f}%, t={t:+.2f}, {(rets > 0).mean()*100:.1f}% up")

baseline = df['next_ret'].dropna()
print(f"\n  Baseline: mean={baseline.mean():+.4f}%, {(baseline > 0).mean()*100:.1f}% up")
print(f"\n  Bonferroni threshold (14 tests): |t| > 2.94")

## Exercises

1. **Try a different asset.** Replace SPY with BTC-USD or GBPUSD=X. Do you discover different archetypes?

2. **Use hourly data.** Fetch `interval="1h"` for intraday patterns. How do they differ from daily?

3. **Add volume.** Include normalised trading volume as a sixth feature. Does it create meaningful new clusters?

4. **Compare with manual labels.** Implement the traditional "hammer" definition (body < 20% of range, lower wick > 60% of range) and cross-tabulate with K-Means clusters.